# Assignment 3: Informed Search

##  Grid Navigation
In this section, you will investigate the problem of navigation on a two dimensional grid with obstacles. The goal is to produce the shortest path between a provided pair of points, taking care to maneuver around the obstacles as needed. Path length is measured in Euclidean distance. Valid directions of movement include up, down, left, right, up-left, up-right, down-left, and down-right. 

### Note:
You are expected to write code where you see **your code here**.  
Make sure you delete the lines with **pass** and **raise NotImplementedError** or your code may not run correctly.

### Task
Your task is to write a function find_path(start, goal, scene) which returns the shortest path from the start point to the goal point that avoids traveling through the obstacles in the grid. 

### Structure/Representation
For this problem, points will be represented as two-element tuples of the form (row, column), and scenes will be represented as two-dimensional lists of Boolean values, with False values corresponding empty spaces and True values corresponding to obstacles. 

### Output
Your output should be the list of points in the path, and should explicitly include both the start point and the goal point. If multiple optimal solutions exist, any of them may be returned. If no optimal solutions exist, or if the start point or goal point lies on an obstacle, you should return the sentinal value None. If start and goal state are the same, return None.

### Implementation
Your implementation should consist of 
* an A* search 
* the straight-line Euclidean distance heuristic. 

Helper functions are allowed and encouraged.

## 1. Euclidean distance
First, let's define a function used for euclidean distance.

In [ ]:
############################################################
# Section 1 Grid Navigation - Euclidiean distance
############################################################
import math

def grid_euclidean_distance(current_state, goal_state):
    return math.sqrt((current_state[0] - goal_state[0]) ** 2 +
                     (current_state[1] - goal_state[1]) ** 2)

In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################

current_state1, goal_state1 = (1,1), (1,1)
assert grid_euclidean_distance(current_state1, goal_state1) == 0

current_state2, goal_state2 = (1,2), (1,1)
assert grid_euclidean_distance(current_state2, goal_state2) == 1

current_state3, goal_state3 = (3,2), (1,2)
assert grid_euclidean_distance(current_state3, goal_state3) == 2

current_state4, goal_state4 = (1,1), (2,2)
assert grid_euclidean_distance(current_state4, goal_state4) == 2**0.5
print("test passed!")

## 2. Helper functions
Next, let's define a functions that finds the successors.

In [ ]:
def grid_successors(current, scene):
    # Order: up, down, left, right, up-left, up-right, down-left, down-right
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1),
                  (-1, -1), (-1, 1), (1, -1), (1, 1)]
    rows = len(scene)
    cols = len(scene[0]) if rows else 0
    result = []
    for dr, dc in directions:
        r, c = current[0] + dr, current[1] + dc
        if 0 <= r < rows and 0 <= c < cols and not scene[r][c]:
            result.append((r, c))
    return tuple(result)

In [ ]:
scene1 = [[True, True, True],
         [False, False, True]]
assert grid_successors((1, 2), scene1) == ((1, 1),)
assert grid_successors((0, 1), scene1) == ((1, 1),(1, 0),)
print("test passed!")

## 3. Find path
Finally let's implement the path search.

In [ ]:
############################################################
# Grid Navigation
############################################################
import collections, itertools, queue, random, copy
import heapq


def find_path(start, goal, scene):
    rows = len(scene)
    cols = len(scene[0]) if rows else 0

    def valid(p):
        return 0 <= p[0] < rows and 0 <= p[1] < cols and not scene[p[0]][p[1]]

    if not valid(start) or not valid(goal) or start == goal:
        return None

    counter = itertools.count()  # tie-breaker so heap never compares tuples/None
    frontier = [(grid_euclidean_distance(start, goal), next(counter), 0.0, start)]
    g_cost = {start: 0.0}
    parent = {start: None}
    closed = set()

    while frontier:
        _, _, g, current = heapq.heappop(frontier)
        if current in closed:
            continue
        if current == goal:
            path = []
            while current is not None:
                path.append(current)
                current = parent[current]
            return path[::-1]
        closed.add(current)

        for nxt in grid_successors(current, scene):
            if nxt in closed:
                continue
            new_g = g + grid_euclidean_distance(current, nxt)
            if nxt not in g_cost or new_g < g_cost[nxt]:
                g_cost[nxt] = new_g
                parent[nxt] = current
                f = new_g + grid_euclidean_distance(nxt, goal)
                heapq.heappush(frontier, (f, next(counter), new_g, nxt))

    return None

In [ ]:
##########################
### TEST YOUR SOLUTION ###
##########################
scene1 = [[False, False, False], 
          [False, True , False], 
          [False, False, False]] 

assert find_path((0, 0), (2, 1), scene1) == [(0, 0), (1, 0), (2, 1)] 

scene2 = [[False, True, False], 
          [False, True, False], 
          [False, True, False]] 
assert find_path((0, 0), (0, 2), scene2) is None
print("test passed!")